# cancer-recon-apoptosis — Step 1: Boltz-2 Oracle Smoke Test (Colab)

**Goal:** verify Boltz-2 separates a real DR5 binder (DR5 ECD + TRAIL ectodomain) from a non-binder (DR5 ECD + scrambled ectodomain), measured by **interface confidence ipTM**.

**Decision rule:** PASS if `ipTM(positive) − ipTM(negative) ≥ 0.15` → proceed to Step 2. Otherwise → pivot per ASSESSMENT.md.

**Why ipTM, not affinity:** Boltz-2's affinity head only scores *small-molecule* ligands; a protein/peptide binder is rejected. For a protein–protein interface, ipTM (range [0,1], higher = more confident interface) is the standard binding proxy — the same metric AlphaFold-Multimer uses.

**Expected time:** 30–60 min total — ~10–15 min for weight download, ~5–10 min per prediction × 2.

**Required runtime:** Colab → Runtime → Change runtime type → **T4 GPU** (or better). Free tier T4 works.

**Idempotency promise:** Every cell is safe to re-run. The script caches per-complex results in `runs/step1_boltz/state.json` and skips any complex whose `confidence_*.json` already exists, so if Colab disconnects mid-way you only re-run what's needed. Each cell starts `[CELL N] ▶` and ends `[CELL N] ✓` so you can see at a glance where a failure was.

## 1. Verify GPU

If `nvidia-smi` shows no GPU, go to *Runtime → Change runtime type → T4 GPU* and rerun.

In [ ]:
#@title Cell 1 — verify GPU
print("[CELL 1] ▶ nvidia-smi")
!nvidia-smi || echo "[CELL 1] ✗ no GPU — change runtime to T4"
print("[CELL 1] ✓ done")

In [ ]:
#@title Cell 2 — verify torch + CUDA
print("[CELL 2] ▶ check torch")
try:
    import torch
    print(f"  torch={torch.__version__}  cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  device={torch.cuda.get_device_name(0)}")
        print(f"  VRAM={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        print("[CELL 2] ✓ done")
    else:
        print("[CELL 2] ✗ no CUDA — runtime is CPU-only; Cell 4 (boltz) will fail")
except Exception as e:
    print(f"[CELL 2] ✗ {type(e).__name__}: {e}")

## 2. Configure repo URL and clone

Set `REPO_URL` to your GitHub repo once you've pushed it. Default below uses the canonical `AnshumanAtrey/cancer-recon-apoptosis` — change if your username is different.

In [ ]:
#@title Cell 3 — clone / pull repo (idempotent)
print("[CELL 3] ▶ clone or pull repo")
import os
from pathlib import Path

# ← EDIT if your GitHub repo is at a different URL
REPO_URL = "https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git"
REPO_DIR = Path("/content/cancer-recon-apoptosis")

try:
    if REPO_DIR.exists():
        print(f"  repo already cloned at {REPO_DIR} — pulling latest")
        !cd {REPO_DIR} && git pull --ff-only
    else:
        print(f"  cloning {REPO_URL}")
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    print(f"  cwd = {os.getcwd()}")
    assert (REPO_DIR / "scripts" / "01_boltz_smoketest.py").exists(), "smoke test script missing"
    print("[CELL 3] ✓ done")
except Exception as e:
    print(f"[CELL 3] ✗ {type(e).__name__}: {e}")
    raise

## 3. Install Boltz-2

First-time install pulls ~10 GB of model weights on first inference (downloaded lazily). The `pip install` itself is fast (~2 min).

In [ ]:
#@title Cell 4 — install Boltz-2 (idempotent — pip skips if already installed)
print("[CELL 4] ▶ pip install boltz biopython")
import shutil, importlib.util
already = shutil.which("boltz") is not None and importlib.util.find_spec("Bio") is not None
if already:
    print("  boltz + biopython already installed — skipping")
else:
    !pip install --quiet boltz biopython
print(f"  boltz CLI: {shutil.which('boltz')}")
print("[CELL 4] ✓ done")

In [ ]:
#@title Cell 5 — verify boltz CLI works
print("[CELL 5] ▶ boltz --help")
import shutil
if shutil.which("boltz") is None:
    print("[CELL 5] ✗ boltz CLI not in PATH — restart runtime and re-run Cell 4")
else:
    !boltz --help | head -15
    print("[CELL 5] ✓ done")

## 4. Run the Step-1 smoke test

This runs `scripts/01_boltz_smoketest.py` which:
1. Loads DR5 ECD, the full TRAIL ectodomain (residues 114–281), and a scrambled TRAIL ectodomain from `data/sequences/`.
2. Writes Boltz YAML inputs for two **protein–protein** complexes: `DR5+TRAIL_ECD` (positive) and `DR5+scrambled` (negative). No affinity property — the binder is a protein, not a small molecule.
3. Invokes `boltz predict --use_msa_server` for each. **First run downloads ~10 GB of weights.**
4. Parses **ipTM** (interface confidence, range [0,1]) from each complex's `confidence_*.json` and reports the margin.

**Decision:** PASS if `ipTM(positive) − ipTM(negative) ≥ 0.15` → the oracle separates a real DR5 binder from noise → proceed to Step 2.

*Why ipTM and not affinity:* Boltz-2's affinity head only scores small-molecule ligands; a protein binder is rejected outright. ipTM is the standard interface-confidence proxy for protein–protein binding (same metric AlphaFold-Multimer uses).

Expected total time: ~15–40 min (two ~300-residue complexes at `diffusion_samples=1`, plus first-run weight download).

In [ ]:
#@title Cell 6 — run the smoke test (resumable: caches per complex)
# Resumability: if a complex already has a confidence JSON it will be SKIPPED.
# To force a full rerun:
#   !rm -rf runs/step1_boltz/state.json runs/step1_boltz/positive runs/step1_boltz/negative
print("[CELL 6] ▶ python -u scripts/01_boltz_smoketest.py")
import subprocess, sys
# Popen + PIPE + re-print via Python's print() — required because
# subprocess.call inherits the OS-level stdout fd which Colab's kernel
# doesn't capture; only Python-level prints reach the cell output.
# `python -u` forces unbuffered stdout in the child so logs stream live.
proc = subprocess.Popen(
    [sys.executable, "-u", "scripts/01_boltz_smoketest.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
exit_code = proc.wait()
print(f"\n[CELL 6] script exit code = {exit_code}")
if exit_code == 0:
    print("[CELL 6] ✓ PASS — oracle discriminates. Proceed to Cell 7/8 to inspect ipTM.")
elif exit_code == 1:
    print("[CELL 6] ✗ FAIL — ipTM margin < 0.15. See pivot options printed above (TRAIL trimer / AF3 / best-of-N).")
elif exit_code == 2:
    print("[CELL 6] ✗ boltz CLI missing. Re-run Cell 4, restart runtime if needed.")
elif exit_code == 3:
    print("[CELL 6] ✗ confidence JSON missing — read the boltz.log tail printed above.")
elif exit_code == 4:
    print("[CELL 6] ✗ FASTA missing — verify data/sequences/ files were cloned (need trail_ecd_human.fasta + scrambled_ecd_control.fasta).")
else:
    print(f"[CELL 6] ✗ unknown exit code {exit_code} — check log above")

## 5. Inspect the raw outputs

Boltz writes per-complex outputs into `runs/step1_boltz/{positive,negative}/`. We look at the affinity JSON and the predicted structure file.

In [ ]:
#@title Cell 7 — inspect raw confidence outputs
print("[CELL 7] ▶ list outputs + display confidence JSONs")
import json
from pathlib import Path

RUN_DIR = Path("runs/step1_boltz")
if not RUN_DIR.exists():
    print(f"[CELL 7] ✗ {RUN_DIR} does not exist — re-run Cell 6")
else:
    print("=== Output tree ===")
    !find runs/step1_boltz -maxdepth 5 -type f | head -50
    print("\n=== Confidence JSONs (ipTM is the binding proxy) ===")
    found = list(RUN_DIR.rglob("confidence_*.json"))
    if not found:
        print("  no confidence JSONs found — boltz may have crashed; read the log tail from Cell 6")
    else:
        for j in sorted(found):
            d = json.loads(j.read_text())
            print(f"\n--- {j.relative_to(RUN_DIR)} ---")
            print(f"  iptm           = {d.get('iptm')}")
            print(f"  ptm            = {d.get('ptm')}")
            print(f"  complex_plddt  = {d.get('complex_plddt')}")
            print(f"  confidence     = {d.get('confidence_score')}")
    print("\n[CELL 7] ✓ done")

In [ ]:
#@title Cell 8 — compute ipTM margin and PASS/FAIL
print("[CELL 8] ▶ load runs/step1_boltz/state.json")
import json
from pathlib import Path

STATE = Path("runs/step1_boltz/state.json")
if not STATE.exists():
    print(f"[CELL 8] ✗ {STATE} missing — re-run Cell 6")
else:
    s = json.loads(STATE.read_text())
    pos = s.get('positive', {})
    neg = s.get('negative', {})
    print(f"  positive (DR5+TRAIL_ECD): status={pos.get('status')} ipTM={pos.get('iptm')} pTM={pos.get('ptm')} plddt={pos.get('complex_plddt')}")
    print(f"  negative (DR5+scrambled): status={neg.get('status')} ipTM={neg.get('iptm')} pTM={neg.get('ptm')} plddt={neg.get('complex_plddt')}")
    margin = s.get('iptm_margin')
    thr = s.get('margin_threshold', 0.15)
    weak = s.get('weak_interface_iptm', 0.40)
    if margin is None:
        print("[CELL 8] ✗ margin not computed — at least one complex failed; re-run Cell 6")
    else:
        print(f"\n  ipTM margin = ipTM(pos) − ipTM(neg) = {margin:+.3f}   (PASS threshold ≥ {thr})")
        if margin >= thr:
            print(f"\n  ✅ PASS — oracle separates DR5 binder from non-binder. Proceed to Step 2.")
            if pos.get('iptm') is not None and float(pos['iptm']) < weak:
                print(f"  ⚠️ note: positive ipTM {float(pos['iptm']):.3f} < {weak} (weak interface).")
                print(f"     It still beats the negative, but for stronger signal model the")
                print(f"     TRAIL homotrimer (3 chains) so the true DR5 groove is present.")
        else:
            print(f"\n  ❌ FAIL — margin < {thr}. See ASSESSMENT.md: try TRAIL trimer / AF3 / best-of-N.")
    print("[CELL 8] ✓ done")

## 6. Save outputs to Google Drive (optional)

Colab sessions are ephemeral — when the runtime resets, `/content/` is wiped. To persist the smoke test results, mount Drive and copy.

In [ ]:
#@title Cell 9 — persist results to Google Drive (optional, idempotent)
print("[CELL 9] ▶ mount Drive + copy runs/step1_boltz/")
from pathlib import Path
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE_OUT = Path('/content/drive/MyDrive/cancer-recon-apoptosis/step1_boltz')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    !cp -r runs/step1_boltz/. '{DRIVE_OUT}/' 2>&1 | tail -5
    print(f"  saved → {DRIVE_OUT}")
    !ls -la '{DRIVE_OUT}' | head -10
    print("[CELL 9] ✓ done")
except ModuleNotFoundError:
    print("[CELL 9] skipped — not running in Colab (no google.colab module)")
except Exception as e:
    print(f"[CELL 9] ✗ {type(e).__name__}: {e}")

## 7. Optional — push results back to GitHub

If you want results committed back to the repo, run the cell below. **Skip if you don't want runs/ in version control** (the project's `.gitignore` already excludes `runs/`).

In [ ]:
# ⚠️ Only run if you want to commit smoke-test results to the repo.
# `.gitignore` excludes runs/ by default — uncomment if you want to override.

# !git config user.name 'Anshuman Atrey'
# !git config user.email 'YOUR_EMAIL@example.com'
# !git add -f runs/step1_boltz/  # -f overrides .gitignore
# !git commit -m 'step 1: Boltz-2 smoke test results'
# !git push  # will prompt for GitHub PAT
print("  (skipped — uncomment cell to push)")

---

## If the smoke test PASSED

1. Update `runs/step1_local/SMOKETEST_RESULTS.md` with the real cloud numbers.
2. Proceed to **Step 2** — CellChat cancer-restricted target shortlist (`scripts/02_cellchat_targets.R`, to be written).

## If the smoke test FAILED (gap < 2 kcal/mol)

See `ASSESSMENT.md` Day-1 kill criteria. Pivot options:
1. **AlphaFold 3 Server** — submit complexes via web API for cross-check.
2. **ABFE pipeline** — Recursion Oct 2025 pipeline atop Boltz-2 ensemble.
3. **OpenMM MD refinement** — 5 ns MD after Boltz-2 prediction for refined ΔG.

## Common Colab issues

- **`boltz: command not found`** → restart runtime after `pip install` and re-run cells from §3.
- **OOM error** → switch to High-RAM runtime, or use `--diffusion_samples 1` instead of 5.
- **MSA server timeout** → re-run; ColabFold MSA queue is sometimes congested. Or pre-compute MSAs locally and pass via `--msa_dir`.
- **Runtime disconnected** → free tier idle limit. Keep tab focused, or upgrade to Colab Pro ($10/mo) for A100.